In [1]:
# Tiny Shakespeare — Build a Transformer

## Objective

# Build a small decoder-only Transformer language model from scratch using PyTorch.

### Architecture

# - Character-level vocabulary
# - Token embeddings
# - Positional embeddings
# - Causal self-attention
# - Multi-head attention
# - Feed-forward network
# - Residual connections
# - Layer normalization
# - Transformer blocks
# - Language-model output head

# The model will predict the next character in a sequence.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
print("PyTorch version:", torch.__version__)
print("MPS available:", torch.backends.mps.is_available())

PyTorch version: 2.14.0
MPS available: True


In [3]:
device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

print("Using device:", device)

Using device: mps


In [4]:
vocab_size = 63
context_length = 128

embedding_dim = 256
num_heads = 4
num_layers = 4

dropout = 0.2

In [5]:
print("Model Configuration")
print("=" * 40)

print("Vocabulary size :", vocab_size)
print("Context length  :", context_length)
print("Embedding dim   :", embedding_dim)
print("Attention heads :", num_heads)
print("Transformer layers:", num_layers)
print("Dropout         :", dropout)

Model Configuration
Vocabulary size : 63
Context length  : 128
Embedding dim   : 256
Attention heads : 4
Transformer layers: 4
Dropout         : 0.2


In [6]:
token_embedding = nn.Embedding(
    vocab_size,
    embedding_dim
)

In [7]:
sample_tokens = torch.tensor(
    [[10, 20, 30, 40]],
    dtype=torch.long
)

embedded = token_embedding(sample_tokens)

print("Input shape:", sample_tokens.shape)
print("Embedding shape:", embedded.shape)

Input shape: torch.Size([1, 4])
Embedding shape: torch.Size([1, 4, 256])


In [8]:
position_embedding = nn.Embedding(
    context_length,
    embedding_dim
)

In [9]:
positions = torch.arange(
    context_length,
    device=device
)

print(positions[:20])

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19], device='mps:0')


In [10]:
x = token_embedding(sample_tokens)

In [11]:
positions = torch.arange(
    sample_tokens.shape[1]
)

pos_emb = position_embedding(positions)

x = x + pos_emb

print(x.shape)

torch.Size([1, 4, 256])


In [12]:
head_dim = embedding_dim // num_heads

print("Head dimension:", head_dim)

Head dimension: 64


In [13]:
class Head(nn.Module):

    def __init__(self, embedding_dim, head_dim, context_length, dropout):
        super().__init__()

        self.key = nn.Linear(
            embedding_dim,
            head_dim,
            bias=False
        )

        self.query = nn.Linear(
            embedding_dim,
            head_dim,
            bias=False
        )

        self.value = nn.Linear(
            embedding_dim,
            head_dim,
            bias=False
        )

        self.dropout = nn.Dropout(dropout)

        self.register_buffer(
            "tril",
            torch.tril(
                torch.ones(
                    context_length,
                    context_length
                )
            )
        )

    def forward(self, x):

        B, T, C = x.shape

        k = self.key(x)
        q = self.query(x)

        attention_scores = q @ k.transpose(-2, -1)

        attention_scores = (
            attention_scores
            * (k.shape[-1] ** -0.5)
        )

        attention_scores = attention_scores.masked_fill(
            self.tril[:T, :T] == 0,
            float("-inf")
        )

        attention_weights = F.softmax(
            attention_scores,
            dim=-1
        )

        attention_weights = self.dropout(
            attention_weights
        )

        v = self.value(x)

        output = attention_weights @ v

        return output

In [14]:
head = Head(
    embedding_dim=embedding_dim,
    head_dim=head_dim,
    context_length=context_length,
    dropout=dropout
)

test_input = torch.randn(
    2,
    10,
    embedding_dim
)

test_output = head(test_input)

print("Input shape :", test_input.shape)
print("Output shape:", test_output.shape)

Input shape : torch.Size([2, 10, 256])
Output shape: torch.Size([2, 10, 64])


In [15]:
class MultiHeadAttention(nn.Module):

    def __init__(
        self,
        embedding_dim,
        num_heads,
        context_length,
        dropout
    ):
        super().__init__()

        head_dim = embedding_dim // num_heads

        self.heads = nn.ModuleList([
            Head(
                embedding_dim,
                head_dim,
                context_length,
                dropout
            )
            for _ in range(num_heads)
        ])

        self.projection = nn.Linear(
            embedding_dim,
            embedding_dim
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        outputs = [
            head(x)
            for head in self.heads
        ]

        x = torch.cat(outputs, dim=-1)

        x = self.projection(x)

        x = self.dropout(x)

        return x

In [16]:
attention = MultiHeadAttention(
    embedding_dim=embedding_dim,
    num_heads=num_heads,
    context_length=context_length,
    dropout=dropout
)

test_input = torch.randn(
    2,
    10,
    embedding_dim
)

test_output = attention(test_input)

print("Input shape :", test_input.shape)
print("Output shape:", test_output.shape)

Input shape : torch.Size([2, 10, 256])
Output shape: torch.Size([2, 10, 256])


In [17]:
class FeedForward(nn.Module):

    def __init__(
        self,
        embedding_dim,
        dropout
    ):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(
                embedding_dim,
                4 * embedding_dim
            ),

            nn.GELU(),

            nn.Linear(
                4 * embedding_dim,
                embedding_dim
            ),

            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.network(x)

In [18]:
ffn = FeedForward(
    embedding_dim,
    dropout
)

test_input = torch.randn(
    2,
    10,
    embedding_dim
)

test_output = ffn(test_input)

print("Input shape :", test_input.shape)
print("Output shape:", test_output.shape)

Input shape : torch.Size([2, 10, 256])
Output shape: torch.Size([2, 10, 256])


In [19]:
class TransformerBlock(nn.Module):

    def __init__(
        self,
        embedding_dim,
        num_heads,
        context_length,
        dropout
    ):
        super().__init__()

        self.ln1 = nn.LayerNorm(
            embedding_dim
        )

        self.attention = MultiHeadAttention(
            embedding_dim,
            num_heads,
            context_length,
            dropout
        )

        self.ln2 = nn.LayerNorm(
            embedding_dim
        )

        self.ffn = FeedForward(
            embedding_dim,
            dropout
        )

    def forward(self, x):

        x = x + self.attention(
            self.ln1(x)
        )

        x = x + self.ffn(
            self.ln2(x)
        )

        return x

In [20]:
block = TransformerBlock(
    embedding_dim,
    num_heads,
    context_length,
    dropout
)

test_input = torch.randn(
    2,
    10,
    embedding_dim
)

test_output = block(test_input)

print("Input shape :", test_input.shape)
print("Output shape:", test_output.shape)

Input shape : torch.Size([2, 10, 256])
Output shape: torch.Size([2, 10, 256])


In [21]:
class MiniGPT(nn.Module):

    def __init__(
        self,
        vocab_size,
        context_length,
        embedding_dim,
        num_heads,
        num_layers,
        dropout
    ):
        super().__init__()

        self.context_length = context_length

        self.token_embedding = nn.Embedding(
            vocab_size,
            embedding_dim
        )

        self.position_embedding = nn.Embedding(
            context_length,
            embedding_dim
        )

        self.blocks = nn.Sequential(
            *[
                TransformerBlock(
                    embedding_dim,
                    num_heads,
                    context_length,
                    dropout
                )
                for _ in range(num_layers)
            ]
        )

        self.ln_final = nn.LayerNorm(
            embedding_dim
        )

        self.lm_head = nn.Linear(
            embedding_dim,
            vocab_size
        )

    def forward(self, idx):

        B, T = idx.shape

        token_embeddings = self.token_embedding(idx)

        positions = torch.arange(
            T,
            device=idx.device
        )

        position_embeddings = self.position_embedding(
            positions
        )

        x = token_embeddings + position_embeddings

        x = self.blocks(x)

        x = self.ln_final(x)

        logits = self.lm_head(x)

        return logits

In [22]:
model = MiniGPT(
    vocab_size=vocab_size,
    context_length=context_length,
    embedding_dim=embedding_dim,
    num_heads=num_heads,
    num_layers=num_layers,
    dropout=dropout
)

model = model.to(device)

print(model)

MiniGPT(
  (token_embedding): Embedding(63, 256)
  (position_embedding): Embedding(128, 256)
  (blocks): Sequential(
    (0): TransformerBlock(
      (ln1): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
      (attention): MultiHeadAttention(
        (heads): ModuleList(
          (0-3): 4 x Head(
            (key): Linear(in_features=256, out_features=64, bias=False)
            (query): Linear(in_features=256, out_features=64, bias=False)
            (value): Linear(in_features=256, out_features=64, bias=False)
            (dropout): Dropout(p=0.2, inplace=False)
          )
        )
        (projection): Linear(in_features=256, out_features=256, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
      )
      (ln2): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
      (ffn): FeedForward(
        (network): Sequential(
          (0): Linear(in_features=256, out_features=1024, bias=True)
          (1): GELU(approximate='none')
          (2

In [23]:
total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print(
    f"Total parameters: "
    f"{total_parameters:,}"
)

Total parameters: 3,221,567


In [24]:
test_batch = torch.randint(
    0,
    vocab_size,
    (2, context_length),
    device=device
)

logits = model(test_batch)

print("Input shape :", test_batch.shape)
print("Logits shape:", logits.shape)

Input shape : torch.Size([2, 128])
Logits shape: torch.Size([2, 128, 63])
